In [1]:
!pip uninstall -y torchao
!pip install -q -U transformers peft accelerate sentence-transformers faiss-cpu datasets huggingface_hub
!git clone https://github.com/gollasudheerbabu09-debug/medical_chatbot.git

import sys, torch
sys.path.insert(0, "medical_chatbot/space")
import rag
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

BASE        = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER     = "Sudheer2002/qwen2.5-0.5b-medquad-qlora"
MERGED_REPO = "Sudheer2002/qwen2.5-0.5b-medquad-merged"
INDEX_REPO  = "Sudheer2002/medquad-faiss-index"

# 1) merge LoRA into the base model
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained("merged_model")
tok = AutoTokenizer.from_pretrained(ADAPTER)
tok.padding_side = "right"
tok.save_pretrained("merged_model")

# 2) build the FAISS index (same split as training, no test answers)
train_df, val_df, test_df = rag.load_splits()
chunks = rag.build_chunks(train_df, exclude_answers=set(test_df["Answer"]))
rag.build_index(chunks, rag.load_embedder(device="cuda"), "rag_index")

# 3) upload both
api = HfApi()
api.create_repo(MERGED_REPO, exist_ok=True)
api.upload_folder(folder_path="merged_model", repo_id=MERGED_REPO)
api.create_repo(INDEX_REPO, repo_type="dataset", exist_ok=True)
api.upload_folder(folder_path="rag_index", repo_id=INDEX_REPO, repo_type="dataset")
print("Merged model:", "https://huggingface.co/" + MERGED_REPO)
print("FAISS index :", "https://huggingface.co/datasets/" + INDEX_REPO)

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 99.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 38.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-clo

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/351 [00:00<?, ?it/s]

Merged model: https://huggingface.co/Sudheer2002/qwen2.5-0.5b-medquad-merged
FAISS index : https://huggingface.co/datasets/Sudheer2002/medquad-faiss-index
